In [27]:
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import HeatMap

place = "District 1, Ho Chi Minh City, Vietnam"
G = ox.graph_from_place(place, network_type='drive')

points = [
    (10.7715, 106.6985, 1.0),
    (10.7702, 106.6830, 0.9),
    (10.7955, 106.6850, 0.8)
]

start = (10.7876, 106.7051)
end = (10.7650, 106.7020)

for u, v, k, data in G.edges(keys=True, data=True):
    data['risk'] = data['length']
    for p in points:
        if abs(G.nodes[u]['y'] - p[0]) < 0.003 and abs(G.nodes[u]['x'] - p[1]) < 0.003:
            data['risk'] = data['length'] * (1 + p[2]*10)

s = ox.distance.nearest_nodes(G, start[1], start[0])
e = ox.distance.nearest_nodes(G, end[1], end[0])

path1 = nx.shortest_path(G, s, e, weight='length')
path2 = nx.shortest_path(G, s, e, weight='risk')

m = folium.Map(location=[10.775,106.700], zoom_start=15)

heat = [[p[0], p[1], p[2]] for p in points]
HeatMap(heat).add_to(m)

coords1 = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in path1]
coords2 = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in path2]

folium.PolyLine(coords1, color='red').add_to(m)
folium.PolyLine(coords2, color='green').add_to(m)

folium.Marker(start).add_to(m)
folium.Marker(end).add_to(m)

m

**Nhận xét:**

Bản đồ thể hiện các khu vực có nguy cơ tắc nghẽn cao bằng heatmap.
Tuyến màu đỏ là đường ngắn nhất nhưng dễ đi qua vùng tắc nghẽn.
Tuyến màu xanh là tuyến được đề xuất, tránh các khu vực rủi ro nên di chuyển ổn định hơn.

Việc sử dụng dữ liệu rủi ro giúp cải thiện lựa chọn đường đi, giảm thời gian di chuyển và hạn chế ùn tắc.